In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
from matplotlib.colors import ListedColormap, BoundaryNorm
import csv
from concave_hull import concave_hull, concave_hull_indexes
from shapely.geometry import Polygon


In [3]:
def import_data_files_for_timeseries_plots(index, key, file_index):

    head = "/Users/dger0683/ChasteWorkspace/chaste_output/New/NoInhibition_strong_100/"

    # import the data file : results.viznodes. 
    # data_rad_file = f"{head}/{key}/MonolayerGrowth_Beta_Index{index}/results_from_time_0/cellareas.dat"
    data_rad_file = f"{head}/{key}/MonolayerGrowth_Beta_Index{index}/results_from_time_0/celldata_Radius.dat"
    if not os.path.isfile(data_rad_file):
        print(f"Data file {data_rad_file} not found.")
        sys.exit(1)

    data_surf_file = f"{head}/{key}/MonolayerGrowth_Beta_Index{index}/results_from_time_0/celldata_FreeSurfaceFraction.dat"
    if not os.path.isfile(data_surf_file):
        print(f"Data file {data_surf_file} not found.")
        sys.exit(1)

    data_area_file = f"{head}/{key}/MonolayerGrowth_Beta_Index{index}/results_from_time_0/celldata_FreeAreaFraction.dat"
    if not os.path.isfile(data_area_file):
        print(f"Data file {data_area_file} not found.")
        sys.exit(1)

    inhibited_growth_file = f"{head}/{key}/MonolayerGrowth_Beta_Index{index}/results_from_time_0/celldata_growth inhibited.dat"
    if not os.path.isfile(inhibited_growth_file):
        print(f"Data file {inhibited_growth_file} not found.")
        sys.exit(1)
    # extract the relavent data from the files
    # open the loggedcell.dat file, and go straight to the last row
    data_list = []
    with open(data_rad_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                row = list(map(float, line.split()))
                data_list.append(row)

    final_row = data_list[file_index][1:]
    # print(f"Final row (excluding time): {final_row}")
    x_vals = final_row[2::5]
    y_vals = final_row[3::5]
    r_vals = final_row[4::5]
    # for i in range(len(r_vals)):
    #     r_vals[i] = np.sqrt(r_vals[i]/np.pi)

    data_surf = []
    with open(data_surf_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                row = list(map(float, line.split()))
                data_surf.append(row)
    surface_fractions = data_surf[file_index][5::5]


    data_area = []
    with open(data_area_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                row = list(map(float, line.split()))
                data_area.append(row)
    area_fractions = data_area[file_index][5::5]

    data_inhibited = []
    with open(inhibited_growth_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                row = list(map(float, line.split()))
                data_inhibited.append(row)
    inhibited_cells = data_inhibited[file_index][5::5]

    time_point = data_list[file_index][0]

    return x_vals, y_vals, r_vals, surface_fractions, area_fractions, inhibited_cells, time_point


In [8]:
# import csv
# # iterate through all indices and collate the data

for index in range(0, 100):
    all_x_vals = []
    all_y_vals = []
    all_r_vals = []
    all_surface_fractions = []
    all_area_fractions = []
    all_inhibited_cells = []
    all_time_points = []
    for file_index in range(250):
        try:
            # x_vals, y_vals, r_vals, surface_fractions, area_fractions, inhibited_cells = import_data_files_for_histograms(index, "100")
            x_vals, y_vals, r_vals, surface_fractions, area_fractions, inhibited_cells, time_point = import_data_files_for_timeseries_plots(index, "quadratic", file_index)
            all_x_vals.extend(x_vals)
            all_y_vals.extend(y_vals)
            all_r_vals.extend(r_vals)
            all_surface_fractions.extend(surface_fractions)
            all_area_fractions.extend(area_fractions)
            all_inhibited_cells.extend(inhibited_cells)
            all_time_points.append(time_point)

            output_file = f'data/CHASTE_MonolayerGrowth_1000_Data_TimeSeries_2/Sim_{index}/cell_data_no_inhibition_{file_index}.csv'
            # check if output folder exists, if not create it
            output_folder = os.path.dirname(output_file)
            if not os.path.exists(output_folder):
                os.makedirs(output_folder)
            with open(output_file, mode='w', newline='') as file:
                writer = csv.writer(file)
                writer.writerow(['x_pos', 'y_pos', 'radius_i', 'f_i', 'a_i'])
                for x, y, r, sf, af in zip(x_vals, y_vals, r_vals, surface_fractions, area_fractions):
                    writer.writerow([x, y, r, sf, af])
            print(f"Data exported to {output_file}")
        except Exception as e:
            print(f"Error occurred while processing file {file_index} for index {index}")
        
    # now we export the time series data for this index
    time_series_output_file = f'data/CHASTE_MonolayerGrowth_1000_Data_TimeSeries_2/Sim_{index}/time_series_data.csv'
    
    with open(time_series_output_file, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['simulation point', 'time_point'])
        for n, t in enumerate(all_time_points):
            writer.writerow([n, t])
    print(f"Time series data exported to {time_series_output_file}")

print(f"Total number of cells across all simulations: {len(all_x_vals)}")
    

Data exported to data/CHASTE_MonolayerGrowth_1000_Data_TimeSeries_2/Sim_0/cell_data_no_inhibition_0.csv
Data exported to data/CHASTE_MonolayerGrowth_1000_Data_TimeSeries_2/Sim_0/cell_data_no_inhibition_1.csv
Data exported to data/CHASTE_MonolayerGrowth_1000_Data_TimeSeries_2/Sim_0/cell_data_no_inhibition_2.csv
Data exported to data/CHASTE_MonolayerGrowth_1000_Data_TimeSeries_2/Sim_0/cell_data_no_inhibition_3.csv
Data exported to data/CHASTE_MonolayerGrowth_1000_Data_TimeSeries_2/Sim_0/cell_data_no_inhibition_4.csv
Data exported to data/CHASTE_MonolayerGrowth_1000_Data_TimeSeries_2/Sim_0/cell_data_no_inhibition_5.csv
Data exported to data/CHASTE_MonolayerGrowth_1000_Data_TimeSeries_2/Sim_0/cell_data_no_inhibition_6.csv
Data exported to data/CHASTE_MonolayerGrowth_1000_Data_TimeSeries_2/Sim_0/cell_data_no_inhibition_7.csv
Data exported to data/CHASTE_MonolayerGrowth_1000_Data_TimeSeries_2/Sim_0/cell_data_no_inhibition_8.csv
Data exported to data/CHASTE_MonolayerGrowth_1000_Data_TimeSerie

In [12]:
# Reprocess all the files titled time_series_data.csv in the CHASTE_MonolayerGrowth_1000_Datas folder.
# In doing so, I want to create a new file titled "tmeline.txt" that contains the points for that given simulation, 
# but scalled by the generation cycle time.

for sim_index in range(100):

    read_csv_file = f'data/CHASTE_MonolayerGrowth_1000_Data_TimeSeries_2/Sim_{sim_index}/time_series_data.csv'
    t = []
    with open(read_csv_file, mode='r') as file:
        reader = csv.reader(file)
        for row in reader:
            if row[1] == 'time_point':
                continue  # Skip the header row
            else:
                t.append(float(row[1]))

    dt = 0.002
    sample_rate = 0.25/dt
    final_area = 2.0
    growth_rate = 0.2
    generation_cycle_time = 0.5*final_area/growth_rate

    t = np.array(t) / generation_cycle_time

    output_file = f'data/CHASTE_MonolayerGrowth_1000_Data_TimeSeries_2/Sim_{sim_index}/timeline.txt'
    with open(output_file, mode='w') as file:
        writer = csv.writer(file)
        # create a header row
        writer.writerow(['index', 'time'])
        for i, time_point in enumerate(t):
            writer.writerow([f"{i} {time_point}"])